# schema-tuned-sql — Remote NVIDIA Tesla T4 Runner

Executes execution-verified SFT (LoRA 1.5B FP16 and QLoRA 3B 4-bit NF4), evaluation on the frozen 220-query test set and 200-query general SQL holdout, and T4 serving benchmarks on an NVIDIA Tesla T4 (sm_75, 16 GB VRAM).

**Hardware Target**: Single NVIDIA Tesla T4 (sm_75, 16 GB VRAM). Precision: FP16 (`float16`), Attention: `sdpa`, Quantization: `4-bit NF4` (QLoRA).

In [ ]:
import os
import subprocess
import torch

# Handle working directory whether repo was cloned or uploaded to /kaggle/working
if os.path.exists("schema-tuned-sql"):
    os.chdir("schema-tuned-sql")

print(f"Working Directory: {os.getcwd()}")
assert torch.cuda.is_available(), "CUDA device not detected! Enable GPU T4 x1 in Kaggle notebook settings."

cc_major, cc_minor = torch.cuda.get_device_capability(0)
device_name = torch.cuda.get_device_name(0)
total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
print(f"GPU: {device_name} (sm_{cc_major}{cc_minor}, {total_mem_gb:.2f} GB VRAM)")

OUT_DIR = "t4_results"
os.makedirs(OUT_DIR, exist_ok=True)
!nvidia-smi

In [ ]:
# Install and upgrade dependencies required for DuckDB execution, TRL, PEFT, and bitsandbytes
!pip install -q duckdb sqlglot trl peft bitsandbytes accelerate pydantic pyyaml

In [ ]:
# Build/verify customer DuckDB warehouse (11 normalized B2B SaaS tables, seed=42)
!python warehouse/generate_data.py --seed 42 --db-path warehouse/analytics.duckdb

# Check that verified 2,000-pair distillation training split is present on disk
import os
assert os.path.exists("data/train/train_2000.jsonl"), "data/train/train_2000.jsonl not found! Check repo upload."
print("Training dataset verified:")
!ls -lh data/train/train_2000.jsonl

In [ ]:
# 1. Fine-Tune S1: Qwen2.5-Coder-1.5B FP16 LoRA (r=16, alpha=32, target_modules q,k,v,o,gate,up,down)
# Expected Runtime on Tesla T4: ~10-14 minutes (2 epochs, batch size 4, grad accum 8)
!python -m src.train.sft --config configs/sft_qwen15b_lora.yaml --seed 42

In [ ]:
# 2. Fine-Tune S2: Qwen2.5-Coder-3B 4-bit QLoRA (NF4, double quant, FP16 compute, r=16, alpha=32)
# Expected Runtime on Tesla T4: ~18-24 minutes (2 epochs, batch size 4, grad accum 8)
!python -m src.train.sft --config configs/sft_qwen3b_qlora.yaml --seed 42

In [ ]:
# 3. Run CUDA Evaluation across Baselines (B0, B2) and Students (S1, S2)
# Evaluates on frozen 220-item test set (EX accuracy, 2000-resample bootstrap CIs, McNemar tests)
# and 200-item cross-domain general SQL holdout (catastrophic forgetting)
# Expected Runtime on Tesla T4: ~12-15 minutes
!python -m src.eval.runner \
    --config configs/eval.yaml \
    --mode cuda_live \
    --systems B0,B2,S1,S2 \
    --out t4_results/student_eval_summary.json

In [ ]:
# 4. Run T4 Serving Latency & Throughput Benchmark
# Measures p50/p95 latency, batch throughput (QPS, tok/s), and peak VRAM across Batch Sizes 1, 8, 32
# Computes cost per 1k queries and break-even volume against frontier API
# Expected Runtime on Tesla T4: ~2-3 minutes
!python -m src.serve.bench \
    --config configs/serve.yaml \
    --mode cuda_live \
    --out t4_results/serving_cost_analysis_t4.json

In [ ]:
# 5. Check generated outputs and package into a downloadable zip file
!ls -lh t4_results/
!zip -r t4_results.zip t4_results/ .artifacts/checkpoints/
print("\n[✓] Telemetry and checkpoints packaged into t4_results.zip. Download this file from the Kaggle sidebar.")